# Tokenization Inspector and Comparison

Language models do not read words directly. A tokenizer converts text into vocabulary entries, then maps those entries to integer token IDs. This notebook compares two Hugging Face tokenizers so that fragmentation, vocabulary differences, special tokens, context usage, and decoding behavior are observable.

We compare:

- `openai-community/gpt2`: an established byte-level BPE tokenizer;
- `Qwen/Qwen2.5-0.5B-Instruct`: a modern multilingual instruct-model tokenizer.

Token IDs are meaningful only to the model trained with the same tokenizer. A lower token count on a few examples does not make one tokenizer universally better.

## 1. Load two tokenizers

`AutoTokenizer` reads each model's tokenizer configuration and constructs the appropriate implementation. `use_fast=True` selects the Rust-backed Hugging Face `tokenizers` implementation when available. Only tokenizer assets are downloaded; no language-model weights are loaded.

In [1]:
import logging
import os
from typing import Any

from transformers import AutoTokenizer, PreTrainedTokenizerBase

os.environ.setdefault("HF_HUB_DISABLE_TELEMETRY", "1")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
logging.getLogger("huggingface_hub").setLevel(logging.ERROR)

TOKENIZER_MODELS = {
    "GPT-2": "openai-community/gpt2",
    "Qwen2.5 Instruct": "Qwen/Qwen2.5-0.5B-Instruct",
}

tokenizers: dict[str, PreTrainedTokenizerBase] = {}
for label, model_name in TOKENIZER_MODELS.items():
    tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)
    tokenizers[label] = tokenizer
    print(
        f"{label}: class={type(tokenizer).__name__}, "
        f"vocabulary={len(tokenizer):,}, fast={tokenizer.is_fast}"
    )

assert len(tokenizers) == 2
assert all(tokenizer.is_fast for tokenizer in tokenizers.values())

GPT-2: class=GPT2Tokenizer, vocabulary=50,257, fast=True


Qwen2.5 Instruct: class=Qwen2Tokenizer, vocabulary=151,665, fast=True


## 2. Define representative inputs

The examples deliberately include text that tokenizers often handle differently: ordinary English, formatted numbers, punctuation and Unicode symbols, source code, Turkish characters and suffixes, and unusually long words.

In [2]:
examples = {
    "English": "Retrieval-augmented generation grounds answers in evidence.",
    "Numbers": "Invoice #2026-0042 totals ₺1,234.56 and 17.5% tax.",
    "Punctuation": "Wait... really?! Email: ai-team@example.com — yes/no.",
    "Code": (
        "def top_k(scores: list[float], k: int = 5) -> list[float]:\n"
        "    return sorted(scores, reverse=True)[:k]"
    ),
    "Turkish": (
        "İstanbul'da geliştirilen bilgi motoru Türkçe soruları güvenilir kaynaklarla yanıtlıyor."
    ),
    "Long words": (
        "antidisestablishmentarianism electroencephalographically "
        "muvaffakiyetsizleştiricileştiriveremeyebileceklerimizdenmişsinizcesine"
    ),
}

for category, text in examples.items():
    print(f"{category:12} characters={len(text):3} whitespace words={len(text.split()):2}")

English      characters= 59 whitespace words= 6
Numbers      characters= 50 whitespace words= 7
Punctuation  characters= 53 whitespace words= 6
Code         characters=102 whitespace words=12
Turkish      characters= 87 whitespace words= 9
Long words   characters=127 whitespace words= 3


## 3. Inspect tokens, IDs, decoding, and special-token handling

A **token** is a vocabulary piece, not necessarily a word. A **token ID** is the integer index of that piece in one tokenizer's vocabulary. IDs cannot be compared semantically across tokenizers. Decoding maps IDs back to text; a correct tokenizer should round-trip these examples without losing content.

In [3]:
def inspect_text(tokenizer: PreTrainedTokenizerBase, text: str) -> dict[str, Any]:
    """Return the observable stages of tokenization for one string."""

    plain = tokenizer(text, add_special_tokens=False, return_attention_mask=False)
    with_special = tokenizer(
        text,
        add_special_tokens=True,
        return_attention_mask=False,
        return_special_tokens_mask=True,
    )
    token_ids = list(plain["input_ids"])
    return {
        "tokens": tokenizer.convert_ids_to_tokens(token_ids),
        "token_ids": token_ids,
        "decoded": tokenizer.decode(token_ids, skip_special_tokens=False),
        "ids_with_special_tokens": list(with_special["input_ids"]),
        "special_tokens_mask": list(with_special["special_tokens_mask"]),
        "token_count": len(token_ids),
    }


inspection_results: dict[str, dict[str, dict[str, Any]]] = {}
for category, text in examples.items():
    inspection_results[category] = {}
    print(f"\n{'=' * 88}\n{category}\nRaw text: {text!r}")
    for label, tokenizer in tokenizers.items():
        result = inspect_text(tokenizer, text)
        inspection_results[category][label] = result
        assert result["decoded"] == text
        print(f"\n[{label}]")
        print("Tokens:             ", result["tokens"])
        print("Token IDs:          ", result["token_ids"])
        print("Decoded text:       ", result["decoded"])
        print("IDs with specials:  ", result["ids_with_special_tokens"])
        print("Special-token mask: ", result["special_tokens_mask"])
        print("Token count:        ", result["token_count"])


English
Raw text: 'Retrieval-augmented generation grounds answers in evidence.'

[GPT-2]
Tokens:              ['Ret', 'ri', 'eval', '-', 'au', 'gment', 'ed', 'Ġgeneration', 'Ġgrounds', 'Ġanswers', 'Ġin', 'Ġevidence', '.']
Token IDs:           [9781, 380, 18206, 12, 559, 5154, 276, 5270, 9384, 7429, 287, 2370, 13]
Decoded text:        Retrieval-augmented generation grounds answers in evidence.
IDs with specials:   [9781, 380, 18206, 12, 559, 5154, 276, 5270, 9384, 7429, 287, 2370, 13]
Special-token mask:  [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
Token count:         13

[Qwen2.5 Instruct]
Tokens:              ['Ret', 'rie', 'val', '-a', 'ug', 'mented', 'Ġgeneration', 'Ġgrounds', 'Ġanswers', 'Ġin', 'Ġevidence', '.']
Token IDs:           [12020, 7231, 831, 7409, 768, 26980, 9471, 20664, 11253, 304, 5904, 13]
Decoded text:        Retrieval-augmented generation grounds answers in evidence.
IDs with specials:   [12020, 7231, 831, 7409, 768, 26980, 9471, 20664, 11253, 304, 5904, 13]
Special-t

`add_special_tokens=True` does not guarantee that tokens will be added to plain text. The model-specific tokenizer decides whether a beginning-of-sequence or end-of-sequence marker belongs in that encoding path. Instruct models commonly apply additional control tokens through a chat template instead.

## 4. Inspect each special-token vocabulary

Special tokens communicate structure rather than normal text content—for example, end of sequence, padding, or chat-message boundaries. Their spelling and IDs are model-specific.

In [4]:
for label, tokenizer in tokenizers.items():
    print(f"\n{label}")
    print("Special-token map:", tokenizer.special_tokens_map)
    print("All special tokens:", tokenizer.all_special_tokens)
    print("All special IDs:   ", tokenizer.all_special_ids)
    print("BOS token / ID:    ", tokenizer.bos_token, tokenizer.bos_token_id)
    print("EOS token / ID:    ", tokenizer.eos_token, tokenizer.eos_token_id)
    print("PAD token / ID:    ", tokenizer.pad_token, tokenizer.pad_token_id)

assert all(tokenizer.eos_token_id is not None for tokenizer in tokenizers.values())


GPT-2
Special-token map: {'bos_token': '<|endoftext|>', 'eos_token': '<|endoftext|>', 'unk_token': '<|endoftext|>'}
All special tokens: ['<|endoftext|>']
All special IDs:    [50256]
BOS token / ID:     <|endoftext|> 50256
EOS token / ID:     <|endoftext|> 50256
PAD token / ID:     None None

Qwen2.5 Instruct
Special-token map: {'eos_token': '<|im_end|>', 'pad_token': '<|endoftext|>'}
All special tokens: ['<|im_end|>', '<|endoftext|>', '<|im_start|>', '<|object_ref_start|>', '<|object_ref_end|>', '<|box_start|>', '<|box_end|>', '<|quad_start|>', '<|quad_end|>', '<|vision_start|>', '<|vision_end|>', '<|vision_pad|>', '<|image_pad|>', '<|video_pad|>']
All special IDs:    [151645, 151643, 151644, 151646, 151647, 151648, 151649, 151650, 151651, 151652, 151653, 151654, 151655, 151656]
BOS token / ID:     None None
EOS token / ID:     <|im_end|> 151645
PAD token / ID:     <|endoftext|> 151643


## 5. Compare fragmentation and token counts

The table below makes the cost and context implication concrete. Character and whitespace-word counts stay fixed, while token counts depend on the vocabulary and segmentation algorithm.

In [5]:
header = (
    f"{'Category':<13} {'Tokenizer':<17} {'Chars':>5} "
    f"{'Words':>5} {'Tokens':>6} {'Chars/token':>11}"
)
print(header)
print("-" * len(header))

count_differences = []
for category, text in examples.items():
    category_counts = []
    for label in tokenizers:
        token_count = inspection_results[category][label]["token_count"]
        category_counts.append(token_count)
        print(
            f"{category:<13} {label:<17} {len(text):>5} {len(text.split()):>5} "
            f"{token_count:>6} {len(text) / token_count:>11.2f}"
        )
    count_differences.append(abs(category_counts[0] - category_counts[1]))

assert any(difference > 0 for difference in count_differences)
largest_difference_index = max(range(len(count_differences)), key=count_differences.__getitem__)
largest_difference_category = list(examples)[largest_difference_index]
print(
    f"\nLargest token-count difference: {largest_difference_category} "
    f"({count_differences[largest_difference_index]} tokens)"
)

Category      Tokenizer         Chars Words Tokens Chars/token
--------------------------------------------------------------
English       GPT-2                59     6     13        4.54
English       Qwen2.5 Instruct     59     6     12        4.92
Numbers       GPT-2                50     7     24        2.08
Numbers       Qwen2.5 Instruct     50     7     32        1.56
Punctuation   GPT-2                53     6     19        2.79
Punctuation   Qwen2.5 Instruct     53     6     14        3.79
Code          GPT-2               102    12     40        2.55
Code          Qwen2.5 Instruct    102    12     29        3.52
Turkish       GPT-2                87     9     42        2.07
Turkish       Qwen2.5 Instruct     87     9     25        3.48
Long words    GPT-2               127     3     42        3.02
Long words    Qwen2.5 Instruct    127     3     36        3.53

Largest token-count difference: Turkish (17 tokens)


## 6. Context-window and cost implication

Models enforce context limits in tokens, not characters or words. For the same nominal context budget, a tokenizer that fragments a particular language or domain more heavily leaves less room for retrieved documents, conversation history, and generated output. API billing also commonly uses token counts.

In this locked run, Qwen used fewer tokens for Turkish (25 vs 42), code (29 vs 40), and punctuation (14 vs 19), while GPT-2 used fewer for the formatted-number example (24 vs 32). This is direct evidence that no tokenizer wins on every input category.

The calculation below is illustrative: it repeats the Turkish sentence and measures how much of an 8,192-token budget the input would consume. It does not claim that both associated models have the same context limit or price.

In [6]:
illustrative_budget = 8_192
repeated_turkish = " ".join([examples["Turkish"]] * 20)

for label, tokenizer in tokenizers.items():
    token_count = len(tokenizer.encode(repeated_turkish, add_special_tokens=False))
    budget_share = 100 * token_count / illustrative_budget
    print(
        f"{label:<17}: {token_count:>5} tokens, "
        f"{budget_share:>6.2f}% of an illustrative {illustrative_budget:,}-token budget"
    )
    assert token_count < illustrative_budget

GPT-2            :   840 tokens,  10.25% of an illustrative 8,192-token budget
Qwen2.5 Instruct :   481 tokens,   5.87% of an illustrative 8,192-token budget


## Observations and acceptance recap

- A token can be a whole word, part of a word, punctuation, whitespace-aware piece, byte sequence, or special control marker.
- Token IDs are vocabulary indexes. The same integer can mean unrelated pieces in different tokenizers.
- Vocabulary size and training data influence fragmentation; neither alone proves downstream model quality.
- Special tokens and chat templates must match the target model.
- Tokenization changes how much text fits into a context window and can change inference cost.
- Decoding verifies that token IDs can be reconstructed into text, although some tokenizers normalize particular inputs.

Questions to answer without rerunning the notebook:

1. Why is a token not the same thing as a word?
2. Why can we not feed GPT-2 token IDs into Qwen?
3. Why might Turkish or source code consume different token counts across tokenizers?
4. How does fragmentation affect context capacity and cost?
5. Why should chat templates be applied by the tokenizer rather than handwritten?